In [20]:
# Step 1: Verify NVIDIA GPU & Install Standard Packages
!nvidia-smi

print("[*] Installing standard Hugging Face packages (Diffusers, Accelerate, FastAPI, Cloudflared)...")
!pip install -q diffusers transformers accelerate trimesh fastapi uvicorn python-multipart

# Download cloudflared tunnel binary for instant public HTTPS endpoint
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("[+] All packages and tools installed successfully in seconds!")

Wed Sep 16 13:54:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P0             29W /   70W |    3031MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import torch
from diffusers import ShapEImg2ImgPipeline
from PIL import Image

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"[*] Loading OpenAI Shap-E model onto: {device}...")

pipe = ShapEImg2ImgPipeline.from_pretrained(
    "openai/shap-e-img2img",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    token=True
).to(device)

gpu_title = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"[+] OpenAI Shap-E Model loaded successfully on: {gpu_title}!")

[*] Loading OpenAI Shap-E model onto: cuda:0...


Couldn't connect to the Hub: 401 Client Error. (Request ID: Root=1-6aaaa074-566766d213786d1b5b883c16;e8899e40-4d37-4628-8f2c-d287e8c8048a)

Repository Not Found for url: https://huggingface.co/api/models/openai/shap-e-img2img.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
User Access Token "all" is expired.
Will try to load from local cache.
The config attributes {'renderer': ['shap_e', 'ShapERenderer']} were passed to ShapEImg2ImgPipeline, but are not expected and will be ignored. Please verify your model_index.json configuration file.
Keyword arguments {'renderer': ['shap_e', 'ShapERenderer']} are not expected by ShapEImg2ImgPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--openai--shap-e-img2img/snapshots/0e0aba80f08d368aaf6af9cb93583707481cc29b/prior: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--openai--shap-e-img2img/snapshots/0e0aba80f08d368aaf6af9cb93583707481cc29b/prior.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--openai--shap-e-img2img/snapshots/0e0aba80f08d368aaf6af9cb93583707481cc29b/shap_e_renderer: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--openai--shap-e-img2img/snapshots/0e0aba80f08d368aaf6af9cb93583707481cc29b/shap_e_renderer.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[+] OpenAI Shap-E Model loaded successfully on: Tesla T4!


In [2]:
# Step 3: Define FastAPI Endpoints (OpenAI Shap-E Pipeline)
import io
import time
import trimesh
from PIL import Image
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from diffusers.utils import export_to_obj

app = FastAPI(title="3D Vision Studio API (OpenAI Shap-E)")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health")
def health_check():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram_alloc = torch.cuda.memory_allocated(0) / (1024 ** 3) if torch.cuda.is_available() else 0.0
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0.0
    return {
        "status": "ok",
        "model": "OpenAI Shap-E (openai/shap-e-img2img)",
        "gpu_name": gpu_name,
        "vram_allocated_gb": round(vram_alloc, 2),
        "vram_total_gb": round(vram_total, 2),
        "model_ready": True,
        "timestamp": time.time()
    }

@app.post("/api/generate")
async def generate_3d(image: UploadFile = File(...)):
    try:
        contents = await image.read()
        pil_img = Image.open(io.BytesIO(contents)).convert("RGB").resize((256, 256))

        # Neural 3D synthesis with OpenAI Shap-E
        images = pipe(
            pil_img,
            guidance_scale=3.0,
            num_inference_steps=64,
            output_type="mesh"
        ).images
        mesh = images[0]

        # Export mesh to temp obj and convert to standard GLB
        temp_obj = "/tmp/shap_e_output.obj"
        export_to_obj(mesh, temp_obj)

        scene = trimesh.load(temp_obj)
        glb_io = io.BytesIO()
        scene.export(glb_io, file_type="glb")

        return Response(
            content=glb_io.getvalue(),
            media_type="model/gltf-binary",
            headers={"Content-Disposition": 'attachment; filename="model.glb"'}
        )
    except Exception as e:
        print(f"[!] Generation error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

print("[+] FastAPI Application defined.")

[+] FastAPI Application defined.


In [4]:
# Step 4: Launch FastAPI Server & Expose via Cloudflare Tunnel
import subprocess
import threading
import time
import re
import uvicorn

# Start Uvicorn in background thread
def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()
time.sleep(2)
print("[+] Uvicorn server listening on port 8000.")

# Launch Cloudflared tunnel
print("[*] Launching secure Cloudflare public tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
for line in iter(tunnel_proc.stdout.readline, ""):
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("\n" + "="*60)
    print("🎉 SUCCESS! YOUR COLAB BACKEND IS ONLINE!")
    print(f"👉 COPY THIS URL INTO YOUR WEB APP:\n{tunnel_url}")
    print("="*60 + "\n")
else:
    print("[!] Cloudflare tunnel did not output URL yet. Check output above.")

[+] Uvicorn server listening on port 8000.
[*] Launching secure Cloudflare public tunnel...

🎉 SUCCESS! YOUR COLAB BACKEND IS ONLINE!
👉 COPY THIS URL INTO YOUR WEB APP:
https://pepper-feat-council-material.trycloudflare.com

